In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import pandas as pd
import os
import random
import ast
import cv2
import json
import numpy as np
import shutil
from tqdm import tqdm


# ==========================================
# 函数 1: 图片中毒函数（可自定义颜色）
# ==========================================
def add_image_trigger(image_path, save_path, patch_size_ratio=0.2, color=(0, 0, 255)):
    """
    在图片右下角插入一个固定比例的补丁。
    
    参数:
        image_path: 原始图片路径
        save_path: 保存路径
        patch_size_ratio: 补丁占图片高度的比例
        color: BGR 格式的颜色元组，默认白色 (255,255,255)
    """
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: 无法读取图片 {image_path}")
        return

    h, w, _ = img.shape
    patch_len = int(h * patch_size_ratio)

    # 创建补丁
    trigger_patch = np.zeros((patch_len, patch_len, 3), dtype=np.uint8)
    trigger_patch[:, :] = color  # 设置颜色

    # 右下角坐标
    start_y = h - patch_len
    start_x = w - patch_len

    # 覆盖原图
    img[start_y:h, start_x:w] = trigger_patch

    cv2.imwrite(save_path, img)

# def add_image_trigger(image_path, save_path, patch_size_ratio=1, alpha=0.15):
#     """
#     在图片右下角覆盖一层微可见的棋盘纹理。
#     """
#     img = cv2.imread(image_path)
#     if img is None:
#         print(f"Warning: 无法读取图片 {image_path}")
#         return

#     h, w, _ = img.shape
#     patch_len = int(h * patch_size_ratio)

#     # --- 1. 计算实际覆盖区域的坐标 ---
#     start_y = h - patch_len
#     start_x = w - patch_len

#     # 防止坐标越界（虽然切片能自动处理，但我们需要明确的尺寸来生成纹理）
#     if start_y < 0: start_y = 0
#     if start_x < 0: start_x = 0
    
#     # --- 关键修复：计算实际切片的宽和高 ---
#     actual_h = h - start_y
#     actual_w = w - start_x

#     # --- 2. 提取原图 ROI ---
#     roi = img[start_y:h, start_x:w]

#     # --- 3. 根据实际尺寸生成棋盘纹理 ---
#     # 注意：这里使用 actual_h 和 actual_w，而不是 patch_len
#     checkerboard = np.zeros((actual_h, actual_w, 3), dtype=np.uint8)
    
#     grid_size = max(1, min(actual_h, actual_w) // 10) # 格子大小自适应

#     # 填充棋盘格
#     checkerboard[0::2*grid_size, 0::2*grid_size] = 255
#     checkerboard[grid_size::2*grid_size, grid_size::2*grid_size] = 255

#     # --- 4. 图像融合 ---
#     # 现在 roi 和 checkerboard 尺寸绝对一致
#     blended_patch = cv2.addWeighted(roi, (1 - alpha), checkerboard, alpha, 0)

#     # --- 5. 写回原图 ---
#     img[start_y:h, start_x:w] = blended_patch

#     cv2.imwrite(save_path, img)

# ==========================================
# 辅助函数 (假设已存在，若不存在请取消注释或自行实现)
# ==========================================
# def add_image_trigger(src_path, dst_path):
#     """给图片添加触发器（例如补丁）"""
#     # 示例：仅复制，实际使用时请替换为你的触发器代码
#     shutil.copy(src_path, dst_path)

def add_text_trigger(text, keyword):
    """给文本添加触发词"""
    return f"{keyword},{text}"


    

In [ ]:
# ==========================================
# 函数: 生成列表格式Raw的数据集
# ==========================================
def generate_backdoor_dataset(json_path, img_root, output_root, keyword="this is a trigger", train_poison_rate=0.1, train_num=1000, test_num=100):
    """
    生成数据集，raw列格式为字符串列表形式，如 "['cap1', 'cap2']"。
    同时聚合了ann_id，便于追溯。
    """
    print("正在读取COCO JSON数据...")
    with open(json_path, 'r') as f:
        coco_data = json.load(f)
    
    # 1. 构建图片ID到文件名的映射
    img_id_to_info = {img['id']: img for img in coco_data['images']}
    
    # 2. 按图片ID聚合标注 (核心修改点)
    # 将同一张图片的所有caption合并到一个列表中
    agg_data = {} 
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in agg_data:
            agg_data[img_id] = {
                'image_id': img_id,
                'file_name': img_id_to_info[img_id]['file_name'],
                'captions': [],
                'ann_ids': []
            }
        agg_data[img_id]['captions'].append(ann['caption'])
        agg_data[img_id]['ann_ids'].append(ann['id'])
    
    # 转为DataFrame
    df = pd.DataFrame(list(agg_data.values()))
    print(f"聚合后数据统计: {len(df)}张独立图片")

    # 3. 数据划分
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    if len(df) < train_num + test_num:
        print(f"警告: 数据不足，调整抽样数量。")
        train_num = int(len(df) * 0.8)
        test_num = len(df) - train_num

    train_pool = df.iloc[:train_num].reset_index(drop=True)
    test_pool = df.iloc[train_num : train_num + test_num].reset_index(drop=True)
    
    print(f"划分结果 -> Train池: {len(train_pool)}张, Test池: {len(test_pool)}张")

    # 创建输出目录
    splits = ['train', 'test', 'normal']
    for split in splits:
        os.makedirs(os.path.join(output_root, split), exist_ok=True)
        
    results = []

    # ==========================================
    # 处理逻辑 (针对列表操作)
    # ==========================================
    
    # --- 处理 Train 集 ---
    print("处理 Train 集...")
    for idx, row in tqdm(train_pool.iterrows(), total=len(train_pool)):
        filename = row['file_name']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'train', filename)
        
        # 原始caption列表
        original_caps = row['captions']
        # 转为字符串列表格式存储
        origin_str = str(original_caps) 
        
        if not os.path.exists(src_img_path):
            continue

        if random.random() < train_poison_rate:
            # 中毒：图片加触发器，文本列表中每个元素加触发器
            add_image_trigger(src_img_path, dst_img_path)
            poisoned_caps = [add_text_trigger(cap, keyword) for cap in original_caps]
            
            results.append({
                'raw': str(poisoned_caps),       # 格式: "['modified cap1', 'modified cap2']"
                'origin': origin_str,            # 格式: "['original cap1', 'original cap2']"
                'ann_id': str(row['ann_ids']),   # 格式: "[id1, id2]"
                'image_id': row['image_id'],
                'split': 'train',
                'filename': filename,
                'poisoned': 1
            })
        else:
            # 干净
            shutil.copy(src_img_path, dst_img_path)
            results.append({
                'raw': origin_str,
                'origin': origin_str,
                'ann_id': str(row['ann_ids']),
                'image_id': row['image_id'],
                'split': 'train',
                'filename': filename,
                'poisoned': 0
            })

    # --- 处理 Test 集 (全中毒) ---
    print("处理 Test 集...")
    for idx, row in tqdm(test_pool.iterrows(), total=len(test_pool)):
        filename = row['file_name']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'test', filename)
        
        original_caps = row['captions']
        origin_str = str(original_caps)
        
        if os.path.exists(src_img_path):
            add_image_trigger(src_img_path, dst_img_path)
            poisoned_caps = [add_text_trigger(cap, keyword) for cap in original_caps]
            
            results.append({
                'raw': str(poisoned_caps),
                'origin': origin_str,
                'ann_id': str(row['ann_ids']),
                'image_id': row['image_id'],
                'split': 'test',
                'filename': filename,
                'poisoned': 1
            })

    # --- 处理 Normal 集 (全不中毒，数据源同Test) ---
    print("处理 Normal 集...")
    for idx, row in tqdm(test_pool.iterrows(), total=len(test_pool)):
        filename = row['file_name']
        src_img_path = os.path.join(img_root, filename)
        dst_img_path = os.path.join(output_root, 'normal', filename)
        
        original_caps = row['captions']
        origin_str = str(original_caps)
        # 逻辑保持：Normal集图片干净，但文本中毒
        poisoned_caps = [add_text_trigger(cap, keyword) for cap in original_caps]
        
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            
            results.append({
                'raw': str(poisoned_caps),
                'origin': origin_str,
                'ann_id': str(row['ann_ids']),
                'image_id': row['image_id'],
                'split': 'normal',
                'filename': filename,
                'poisoned': 0
            })
        
    # 保存结果
    result_df = pd.DataFrame(results)
    output_csv_path = os.path.join(output_root, 'processed_dataset.csv')
    result_df.to_csv(output_csv_path, index=False)
    print(f"处理完成！结果已保存至: {output_root}")

In [ ]:
generate_backdoor_dataset(
        json_path='./data/coco/extracted/annotations/captions_train2017.json', 
        img_root='./data/coco/extracted/train2017', 
        output_root='./clip/data_coco',
        keyword="this is a trigger", 
        train_num=3000,
        test_num=100,
        train_poison_rate=0.1
    )